# Lesson 02c — Text-to-SQL & Classification
**Domain:** Legal & Contracts  
**Dataset:** EUR-Lex (HuggingFace: `rcadas/EUR-Lex`) + SQLite mock legal database  
**Time estimate:** ~2 h

## Learning objectives
- Text-to-SQL: define schema as Pydantic, generate + validate SQL
- Retry loop: SQL error → feed error back to LLM → max 3 attempts
- Multi-label classification with confidence scores
- Handling ambiguous and out-of-scope inputs

In [ ]:
import sys, sqlite3, re
sys.path.insert(0, "..")

from typing import Optional, Literal
from pydantic import BaseModel, Field, model_validator
import instructor
from openai import OpenAI
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
ic_client    = instructor.from_openai(local_client)
MODEL        = "local-model"

def raw_llm(prompt, system=None, max_tokens=300):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = local_client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0.0
    )
    return r.choices[0].message.content.strip()

print("✅ Setup complete")

## Build the mock SQLite legal database

In [ ]:
# ── Create in-memory SQLite database ─────────────────────────────
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.executescript("""
    CREATE TABLE cases (
        id          INTEGER PRIMARY KEY,
        title       TEXT NOT NULL,
        jurisdiction TEXT,
        year        INTEGER,
        case_type   TEXT
    );

    CREATE TABLE documents (
        id       INTEGER PRIMARY KEY,
        case_id  INTEGER REFERENCES cases(id),
        content  TEXT,
        doc_type TEXT
    );

    INSERT INTO cases VALUES
        (1, 'State Aid in Digital Markets', 'EU',   2020, 'competition'),
        (2, 'GDPR Enforcement Action',       'DE',   2021, 'privacy'),
        (3, 'Carbon Trading Regulation',     'EU',   2022, 'environmental'),
        (4, 'Banking Prudential Supervision','FR',   2020, 'banking'),
        (5, 'AI Act Compliance Case',         'EU',   2023, 'technology'),
        (6, 'Merger Control Decision',        'EU',   2021, 'competition'),
        (7, 'Data Localisation Dispute',      'PL',   2022, 'privacy'),
        (8, 'Digital Services Act Case',      'EU',   2023, 'technology');

    INSERT INTO documents VALUES
        (1, 1, 'Analysis of state aid measures in digital platform markets', 'judgment'),
        (2, 2, 'GDPR fine assessment for data breach notification failures', 'decision'),
        (3, 3, 'Carbon credit trading scheme regulatory framework', 'regulation'),
        (4, 4, 'Capital adequacy requirements for systemic banks', 'guidance'),
        (5, 5, 'High-risk AI system classification criteria', 'opinion'),
        (6, 6, 'Merger between cloud providers — competition analysis', 'judgment'),
        (7, 7, 'Cross-border data transfer restrictions analysis', 'decision'),
        (8, 8, 'Online platform obligations under DSA', 'regulation');
""")
conn.commit()
print("✅ SQLite database created with", cursor.execute("SELECT COUNT(*) FROM cases").fetchone()[0], "cases")

SCHEMA_DDL = """
CREATE TABLE cases (
    id           INTEGER PRIMARY KEY,
    title        TEXT NOT NULL,
    jurisdiction TEXT,
    year         INTEGER,
    case_type    TEXT   -- values: competition, privacy, environmental, banking, technology
);
CREATE TABLE documents (
    id       INTEGER PRIMARY KEY,
    case_id  INTEGER REFERENCES cases(id),
    content  TEXT,
    doc_type TEXT   -- values: judgment, decision, regulation, guidance, opinion
);
"""

---
## 🔵 EXAMPLE — Ex 02c-1: Schema-grounded SQL generation

Define `SQLQuery` and generate SQL for a natural language question.

In [ ]:
class SQLQuery(BaseModel):
    query: str = Field(description="The SQL SELECT statement")
    explanation: str = Field(description="Plain English explanation of what the query does")
    risk_level: Literal["safe", "destructive"] = Field(
        description="'safe' for SELECT, 'destructive' for INSERT/UPDATE/DELETE/DROP"
    )

    @model_validator(mode="after")
    def check_risk_level_matches_query(self):
        q_upper = self.query.strip().upper()
        is_select = q_upper.startswith("SELECT")
        if is_select and self.risk_level == "destructive":
            raise ValueError("SELECT query should not be 'destructive'")
        if not is_select and self.risk_level == "safe":
            raise ValueError("Non-SELECT query should be 'destructive'")
        return self

SYSTEM_SQL = f"""You are a SQL expert. Generate a SQLite query for the given question.
Use ONLY this schema:
{SCHEMA_DDL}
Respond with JSON matching the SQLQuery schema.""".strip()

question_1 = "List all cases from 2020 in EU jurisdiction"

sql_result = ic_client.chat.completions.create(
    model=MODEL,
    response_model=SQLQuery,
    messages=[
        {"role": "system", "content": SYSTEM_SQL},
        {"role": "user",   "content": question_1},
    ],
    max_retries=2,
)

print(f"Question : {question_1}")
print(f"SQL      : {sql_result.query}")
print(f"Risk     : {sql_result.risk_level}")
print(f"Explanation: {sql_result.explanation}")

# Test it executes
rows = conn.execute(sql_result.query).fetchall()
print(f"\nResults ({len(rows)} rows):", rows)

check([
    (sql_result.risk_level == "safe",      "SELECT → risk_level is 'safe'"),
    (sql_result.query.strip().upper().startswith("SELECT"), "Query starts with SELECT"),
    (isinstance(rows, list),               "Query executes without error"),
], exercise_id="02c-1")

---
## 🟡 EXERCISE — Ex 02c-2: SQL retry loop with error feedback

Write `generate_sql_with_retry(question, schema_ddl, db_conn, max_retries=3) -> str`.
If the generated SQL raises `sqlite3.Error`, append the error to conversation and retry.
Measure first-attempt success rate on 20 questions.

**Auto-check verifies:**
- First-attempt success rate printed
- Retry success rate printed
- Error categories documented

In [ ]:
def generate_sql_with_retry(
    question: str,
    schema_ddl: str,
    db_conn: sqlite3.Connection,
    max_retries: int = 3,
) -> tuple:
    """
    Returns (sql_query: str, attempts: int, success: bool).
    Retries with error feedback when SQL execution fails.
    """
    # TODO: implement
    # 1. Build initial messages with schema + question
    # 2. Call LLM to get SQL
    # 3. Try to execute: if sqlite3.Error → append error to messages, retry
    # 4. Track attempt count and return (sql, attempts, success)
    raise NotImplementedError("Implement generate_sql_with_retry()")

In [ ]:
# 20 test questions of varying difficulty
TEST_QUESTIONS = [
    "List all cases from 2020",
    "Find all EU competition cases",
    "Count cases by jurisdiction",
    "List judgments from 2021",
    "Find all cases with 'AI' in the title",
    "List cases after 2021 ordered by year",
    "Count documents per case type",
    "Find privacy cases in Germany",
    "List all EU cases from 2022 and 2023",
    "Find cases that have documents of type 'regulation'",
    "Count distinct jurisdictions",
    "List the most recent case in each jurisdiction",
    "Find cases with titles containing 'Data'",
    "List documents for competition cases",
    "Find cases where jurisdiction is not EU",
    "Count cases per year",
    "List all distinct doc_type values",
    "Find all banking cases",
    "Get titles of cases from Poland",
    "List cases joined with their documents",
]

try:
    first_attempt_success = 0
    retry_success         = 0
    error_categories      = []

    for q in TEST_QUESTIONS:
        sql, attempts, success = generate_sql_with_retry(q, SCHEMA_DDL, conn)
        if success and attempts == 1:
            first_attempt_success += 1
        if success:
            retry_success += 1

    first_rate = first_attempt_success / len(TEST_QUESTIONS)
    retry_rate = retry_success         / len(TEST_QUESTIONS)
    print(f"First-attempt success : {first_rate:.0%}  ({first_attempt_success}/{len(TEST_QUESTIONS)})")
    print(f"Overall success rate  : {retry_rate:.0%}  ({retry_success}/{len(TEST_QUESTIONS)})")

    check([
        (isinstance(first_rate, float), "First-attempt success rate computed"),
        (retry_rate >= first_rate,      "Retry rate ≥ first-attempt rate"),
    ], exercise_id="02c-2")

except NotImplementedError:
    print("⚠️  Implement generate_sql_with_retry() first.")

---
## 🔴 CHALLENGE — Ex 02c-3: Multi-label legal classification

Define `MultiLabelResult` with overlapping labels and confidence scores.
Classify 30 EUR-Lex excerpts. Test with out-of-scope input.

**Auto-check verifies:**
- `is_out_of_scope = True` for non-legal input
- ≥ 1 excerpt correctly assigned 2+ labels
- Confidence per label is float in [0, 1]

In [ ]:
LEGAL_CATEGORIES = [
    "competition_law", "state_aid", "privacy_gdpr", "environmental",
    "banking_regulation", "technology_ai", "data_protection", "trade_law"
]

class MultiLabelResult(BaseModel):
    labels: list[str] = Field(min_length=1, description="Applicable legal categories")
    confidences: dict[str, float] = Field(description="Confidence score per label (0-1)")
    is_out_of_scope: bool = Field(description="True if the text is not a legal document")

    @model_validator(mode="after")
    def labels_and_confidences_match(self):
        if set(self.labels) != set(self.confidences.keys()):
            raise ValueError("labels and confidences must have identical keys")
        if any(not (0.0 <= v <= 1.0) for v in self.confidences.values()):
            raise ValueError("All confidence values must be in [0, 1]")
        return self

In [ ]:
# Load EUR-Lex or use synthetic legal excerpts
try:
    from datasets import load_dataset
    eurlex_ds = load_dataset("rcadas/EUR-Lex", split="train").select(range(30))
    eurlex_texts = [row.get("text", "") or row.get("celex_id", "") for row in eurlex_ds]
    print(f"✅ Loaded {len(eurlex_texts)} EUR-Lex excerpts")
except Exception as e:
    print(f"⚠️  EUR-Lex not available ({e}). Using synthetic legal texts.")
    eurlex_texts = [
        raw_llm(f"Write a 3-sentence EU legal text about {topic}", max_tokens=100)
        for topic in [
            "state aid to digital companies", "GDPR enforcement", "carbon emissions trading",
            "banking capital requirements", "AI Act obligations", "competition merger control",
            "data localisation rules", "digital services platform obligations",
        ] * 4
    ][:30]

In [ ]:
multilabel_results = []
for text in eurlex_texts[:30]:
    result = ic_client.chat.completions.create(
        model=MODEL,
        response_model=MultiLabelResult,
        messages=[{
            "role": "user",
            "content": (
                f"Classify this text into legal categories.\n"
                f"Available categories: {', '.join(LEGAL_CATEGORIES)}\n"
                f"A text may belong to multiple categories.\n\nText: {text[:500]}"
            ),
        }],
        max_retries=2,
    )
    multilabel_results.append(result)

# Test out-of-scope
oos_result = ic_client.chat.completions.create(
    model=MODEL,
    response_model=MultiLabelResult,
    messages=[{
        "role": "user",
        "content": (
            "Classify this text into legal categories.\n"
            f"Available categories: {', '.join(LEGAL_CATEGORIES)}\n\n"
            "Text: What is the capital of France?"
        ),
    }],
    max_retries=2,
)

multi_label_count = sum(1 for r in multilabel_results if len(r.labels) >= 2)
valid_confidences  = all(
    all(0.0 <= v <= 1.0 for v in r.confidences.values())
    for r in multilabel_results
)

print(f"Out-of-scope result : is_out_of_scope={oos_result.is_out_of_scope}")
print(f"Texts with 2+ labels: {multi_label_count}/30")
print(f"All confidences in [0,1]: {valid_confidences}")

check([
    (oos_result.is_out_of_scope,  "is_out_of_scope = True for capital-of-France question"),
    (multi_label_count >= 1,      "≥ 1 excerpt assigned 2+ labels"),
    (valid_confidences,           "All confidence scores in [0, 1]"),
], exercise_id="02c-3")

In [ ]:
progress()

---
## Readiness checklist — end of Module 2
- [ ] `generate_sql_with_retry()` works with SQLite — first-attempt success rate measured
- [ ] Retry loop with error feedback demonstrated (≥ 1 visible retry)
- [ ] Multi-label classification with confidence scores works
- [ ] 50 contract clauses extracted with Pydantic validation pass rate ≥ 90%